# Agentic AI Project

I'm running this project to be able to better understand agents, using Kaggle's course on AI Agents to get a fundamental grasp of them.

Link to course: https://www.kaggle.com/learn-guide/5-day-agents

In [23]:
import os
from dotenv import load_dotenv
import requests
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types
from google.adk.models.lite_llm import LiteLlm

In [2]:
load_dotenv() 
api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("Authentication Error: GOOGLE_API_KEY not found in environment.")
print("Gemini API key setup complete.")

Gemini API key setup complete.


In [3]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry when facing a specific HTTP error
)

In [4]:
# List of tools: https://adk.dev/integrations/
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)
# Agent in charge of orchestration

print("Root Agent defined.")

Root Agent defined.


In [5]:
runner = InMemoryRunner(agent=root_agent)

In [6]:
question = "What is the weather like in Paris today?"
response = await runner.run_debug(question)

# Web UI to see agent reasoning

helpful_assistant > The weather in Paris today, May 26, 2026, is sunny with a high of 92°F (33°C) and a low of 61°F (16°C). The humidity is around 41%, and the wind is from the north at 3 mph. There is a 0% chance of precipitation. Some sources indicate the high temperature could reach up to 90-92°F (32-33°C). The "feels like" temperature is also around 90-92°F (32-33°C).


In [7]:
# !adk create sample-agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY
# Creating the environment

In [10]:
!adk web sample-agent --port 8000


"""
http://127.0.0.1:8000

If a port is stuck, input the following command in PowerShell to free it:
Stop-Process -Id (Get-NetTCPConnection -LocalPort 8000 -State Listen -ErrorAction SilentlyContinue).OwningProcess -Force


"""

^C


In [11]:
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings"
)

In [ ]:
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

In [15]:
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)


In [16]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)

c:\Users\Nicolas\OneDrive - SKEMA Business School\FMI\Projects\Agentic Project\.venv\Lib\site-packages\google\adk\models\llm_request.py:256: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


ResearchCoordinator > Quantum computing is set to revolutionize AI by significantly boosting computational power. This advancement will enable AI models to train much faster and with greater accuracy, allowing them to tackle complex problems that are currently beyond our reach. New applications in fields like drug discovery and climate modeling will emerge, and AI systems are expected to become more reliable with reduced uncertainty. Major companies like IBM and Google are leading the development in this space, and by 2030, the integration of quantum computing and AI is poised to drive substantial innovation and accelerate AI's impact across various sectors.


# Multi-Agent Architecture

In [17]:
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    print("Ollama up. Models:", [m["name"] for m in r.json().get("models", [])])
except Exception as e:
    print("Ollama not reachable:", e)

Ollama up. Models: ['qwen3.6:latest']


In [26]:
# 1. Local Ollama Qwen model
# "ollama/" tells LiteLLM to route to your local Ollama instance.
# The tag after it must match `ollama list` exactly.
local_qwen = LiteLlm(
    model="ollama/qwen3.6:latest",
    api_base="http://localhost:11434",
)

# 2. Summarization agent
summarizer_agent2 = Agent(
    name="summarizer",
    model=local_qwen,
    description="An agent specialized in summarizing documents and text.",
    instruction="""You are an expert summarizer.
Analyze the text provided by the user and produce a concise summary with:
1. A single-sentence executive summary.
2. A bulleted list of the most important takeaways (maximum 5 bullets).
3. Do not add any extra filler text; keep the tone professional and objective.""",
)

In [27]:
root_agent2 = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent2)],
)


In [28]:
runner = InMemoryRunner(agent=root_agent2)
response = await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)

ResearchCoordinator > The latest advancements in quantum computing are set to revolutionize AI by dramatically increasing computational power and enabling the solution of complex problems that are currently intractable for classical computers. This synergy, known as Quantum AI, promises to significantly enhance AI's capabilities in several ways:

*   **Increased Computational Power:** Quantum computers use qubits, which can represent multiple states simultaneously, allowing for massive parallel processing. This can accelerate AI tasks like database searches.
*   **Faster AI Model Training:** Quantum computing can drastically reduce the time needed to train complex AI models, leading to more accurate and responsive AI systems.
*   **Improved AI Accuracy and Capabilities:** Quantum computers can perform complex simulations and optimizations, leading to more nuanced and accurate AI predictions. This could accelerate breakthroughs in fields like drug discovery.
*   **New AI Algorithms:** Q